# EINX — Train + Test on Free GPU

**What to do:**
1. Click **Runtime** → **Change runtime type** → **T4 GPU** → Save
2. Click **Runtime** → **Run all** (or Ctrl+F9)
3. Wait ~10 minutes
4. Scroll to bottom to see the test results

In [ ]:
# ============================================================
# CELL 1: Setup — clone repo, install deps, verify GPU
# ============================================================
import os, sys, subprocess

# Clone repo
if not os.path.exists('/content/einx'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/lewiseinstein15-Tech/einx-fooundation-model.git',
                    '/content/einx'], check=True)

sys.path.insert(0, '/content/einx')

# Install deps (Colab already has torch)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pydantic', 'pyyaml', 'requests'], check=False)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU! Go to Runtime -> Change runtime type -> GPU')
    DEVICE = 'cpu'

# Test that EINX imports work
from einx.tokenizer.bpe import BPETokenizer
from einx.model.transformer import EINXTransformer
from einx.training.trainer import EINXTrainer
from einx.data.shards import ShardDataset, ShardWriter
from einx.data.dataset import load_jsonl, train_val_test_split
print('EINX imports OK')

In [ ]:
# ============================================================
# CELL 2: Generate knowledge corpus + train tokenizer + build dataset
# ============================================================
import json, random, hashlib
from pathlib import Path

# --- Knowledge corpus (same as generate_knowledge_v2.py, inlined) ---
CAPITALS = [
    ('france','paris'),('japan','tokyo'),('england','london'),('germany','berlin'),
    ('italy','rome'),('china','beijing'),('russia','moscow'),('india','new delhi'),
    ('brazil','brasilia'),('egypt','cairo'),('kenya','nairobi'),('canada','ottawa'),
    ('australia','canberra'),('spain','madrid'),('greece','athens'),('portugal','lisbon'),
    ('netherlands','amsterdam'),('sweden','stockholm'),('norway','oslo'),('finland','helsinki'),
    ('denmark','copenhagen'),('poland','warsaw'),('turkey','ankara'),('south korea','seoul'),
    ('mexico','mexico city'),('argentina','buenos aires'),('chile','santiago'),('peru','lima'),
    ('thailand','bangkok'),('vietnam','hanoi'),('indonesia','jakarta'),
    ('saudi arabia','riyadh'),('iran','tehran'),('iraq','baghdad'),
    ('switzerland','bern'),('austria','vienna'),('belgium','brussels'),('ireland','dublin'),
]

SCIENCE_FACTS = [
    'water boils at 100 degrees celsius.','water freezes at 0 degrees celsius.',
    'the earth orbits the sun.','the moon orbits the earth.','the sun is a star.',
    'gravity pulls objects toward the earth.','a year has 365 days.','a week has 7 days.',
    'a day has 24 hours.','an hour has 60 minutes.','a minute has 60 seconds.',
    'there are 12 months in a year.','january is the first month.','december is the last month.',
    'humans have 206 bones.','the human heart has 4 chambers.','blood is red because of hemoglobin.',
    'plants make food through photosynthesis.','photosynthesis converts sunlight into energy.',
    'the largest planet is jupiter.','the smallest planet is mercury.','mars is called the red planet.',
    'venus is the hottest planet.','the pacific ocean is the largest ocean.',
    'mount everest is the tallest mountain.','a triangle has 3 sides.','a square has 4 equal sides.',
    'a circle has no corners.','a pentagon has 5 sides.','a hexagon has 6 sides.','an octagon has 8 sides.',
    'there are 8 planets in the solar system.','the earth has one moon.','mars has two moons.',
    'saturn has rings made of ice and rock.','the human body has 5 senses.',
    'the brain is the control center of the body.','the lungs are used for breathing.',
    'the heart pumps blood through the body.','the skin is the largest organ.',
    'diamond is the hardest natural material.','gold does not rust.',
    'a magnet has a north pole and a south pole.','opposite poles attract.','like poles repel.',
    'iron is heavier than wood.','a whale is bigger than a mouse.','a cheetah is faster than a turtle.',
]

LOGIC = [
    'if a is bigger than b, and b is bigger than c, then a is bigger than c.',
    'if all cats are animals, and tom is a cat, then tom is an animal.',
    'if all birds have wings, and a robin is a bird, then a robin has wings.',
    'if all fish live in water, and a salmon is a fish, then salmon live in water.',
    'if it is raining, the ground gets wet.',
    'if x is greater than 5, and 5 is greater than 3, then x is greater than 3.',
    'if all squares have 4 sides, and this shape has 3 sides, then it is not a square.',
    'if a number is even, it is divisible by 2. 8 is even, so 8 is divisible by 2.',
    'if a number is odd, it is not divisible by 2. 7 is odd, so 7 is not divisible by 2.',
    'if today is monday, tomorrow is tuesday.','if today is tuesday, tomorrow is wednesday.',
    'if today is wednesday, tomorrow is thursday.','if today is thursday, tomorrow is friday.',
    'if today is friday, tomorrow is saturday.','if today is saturday, tomorrow is sunday.',
    'if today is sunday, tomorrow is monday.','if today is monday, yesterday was sunday.',
    'if today is friday, yesterday was thursday.','if a + b = 10 and a = 3, then b = 7.',
    'if 2x = 10, then x = 5.','if 3x = 15, then x = 5.','if x + 5 = 12, then x = 7.',
    'if x + 3 = 10, then x = 7.','if x - 4 = 6, then x = 10.',
    'if a shape has 3 sides, it is a triangle.','if a shape has 4 equal sides, it is a square.',
    'if all mammals have hair, and a whale is a mammal, then whales have hair.',
]

DEFINITIONS = [
    'a mammal is an animal that has hair and feeds its young with milk.',
    'a reptile is an animal with scales that lays eggs.','a bird is an animal with feathers and a beak.',
    'a fish is an animal that lives in water and has gills.','an insect is an animal with 6 legs and 3 body parts.',
    'gravity is the force that pulls objects toward each other.','friction opposes motion when two surfaces touch.',
    'energy is the ability to do work.','a molecule is two or more atoms joined together.',
    'an atom is the smallest unit of matter.','a cell is the basic unit of life.',
    'dna carries genetic information.','a planet orbits a star.','a star is a ball of hot gas that produces light.',
    'a galaxy is a group of stars.','temperature measures how hot or cold something is.',
    'mass is the amount of matter in an object.','volume is the amount of space something takes up.',
    'density is mass divided by volume.','speed is distance divided by time.',
    'a solid has a fixed shape.','a liquid takes the shape of its container.','a gas fills its container.',
    'a herbivore eats only plants.','a carnivore eats only meat.','an omnivore eats both plants and meat.',
    'a predator hunts other animals.','a habitat is the natural home of an animal.',
]

COMMONSENSE = [
    'fire is hot.','ice is cold.','the sky is blue during the day.','the sky is dark at night.',
    'you should drink water when thirsty.','you should eat food when hungry.','you should sleep when tired.',
    'wear warm clothes in winter.','wear light clothes in summer.','look both ways before crossing the street.',
    'wash your hands before eating.','brush your teeth to keep them healthy.','rain makes the ground wet.',
    'sun makes things warm.','wind can blow things away.','snow is cold and white.','a knife is sharp.',
    'a pillow is soft.','a rock is hard.','a feather is light.','you need air to breathe.',
    'you need water to live.','you need food to live.','plants need sunlight to grow.','fish live in water.',
    'birds live in trees.','humans walk on two legs.','dogs walk on four legs.','spiders have eight legs.',
    'insects have six legs.','the sun gives us light and heat.','the moon shines at night.',
]

CAUSE_EFFECT = [
    'if you heat ice, it melts into water.','if you heat water, it boils and becomes steam.',
    'if you cool water, it freezes into ice.','if you drop something, gravity pulls it down.',
    'if you mix red and blue paint, you get purple.','if you mix yellow and blue paint, you get green.',
    'if you mix red and yellow paint, you get orange.','if you touch something hot, you will burn your hand.',
    'if you do not eat, you will lose weight.','if you exercise, your muscles get stronger.',
    'if you study, you learn.','if you do not drink water, you become thirsty.',
    'if the sun shines, things become warm.','if wind blows, leaves move.',
    'if you leave metal in water, it may rust.','if you plant a seed and water it, it grows.',
    'if you stay in the sun too long, you may get sunburned.','if you do not brush your teeth, you may get cavities.',
]

REASONING = [
    'problem: if you have 5 apples and eat 2, how many are left? solution: 5 - 2 = 3. answer: 3 apples.',
    'problem: if you have 10 dollars and buy a toy for 3 dollars, how much is left? solution: 10 - 3 = 7. answer: 7 dollars.',
    'problem: if you have 2 red balls and 3 blue balls, how many balls? solution: 2 + 3 = 5. answer: 5 balls.',
    'problem: if each box holds 6 eggs and you have 4 boxes, how many eggs? solution: 6 x 4 = 24. answer: 24 eggs.',
    'problem: if a pizza has 8 slices and you eat 3, how many left? solution: 8 - 3 = 5. answer: 5 slices.',
    'problem: if you read 10 pages a day for 5 days, how many pages? solution: 10 x 5 = 50. answer: 50 pages.',
    'problem: if you have 20 dollars and pencils cost 2 dollars each, how many pencils? solution: 20 / 2 = 10. answer: 10 pencils.',
    'problem: if a is 5 and b is 3, what is a + b? solution: 5 + 3 = 8. answer: 8.',
    'problem: if a is 5 and b is 3, what is a - b? solution: 5 - 3 = 2. answer: 2.',
    'problem: if a is 5 and b is 3, what is a x b? solution: 5 x 3 = 15. answer: 15.',
    'problem: if 3 people share 12 cookies equally, how many each? solution: 12 / 3 = 4. answer: 4 cookies.',
    'problem: if today is wednesday, what day was yesterday? answer: tuesday.',
    'problem: if today is thursday, what day is tomorrow? answer: friday.',
]

def generate_corpus(n=30000, seed=42):
    rng = random.Random(seed)
    records = []
    for _ in range(n):
        r = rng.random()
        if r < 0.15:
            a, b = rng.randint(1,12), rng.randint(1,12)
            op = rng.choice(['+','-','x'])
            if op == '+': records.append({'text': f'{a} + {b} = {a+b}.'})
            elif op == '-': a,b = max(a,b),min(a,b); records.append({'text': f'{a} - {b} = {a-b}.'})
            else: records.append({'text': f'{a} x {b} = {a*b}.'})
        elif r < 0.35:
            if rng.random() < 0.5:
                c, cap = rng.choice(CAPITALS); records.append({'text': f'the capital of {c} is {cap}.'})
            else:
                records.append({'text': rng.choice(SCIENCE_FACTS)})
        elif r < 0.45:
            records.append({'text': rng.choice(LOGIC)})
        elif r < 0.55:
            records.append({'text': rng.choice(DEFINITIONS)})
        elif r < 0.65:
            records.append({'text': rng.choice(COMMONSENSE)})
        elif r < 0.75:
            records.append({'text': rng.choice(CAUSE_EFFECT)})
        elif r < 0.85:
            records.append({'text': rng.choice(REASONING)})
        else:
            # Q&A pairs
            if rng.random() < 0.3:
                c, cap = rng.choice(CAPITALS)
                records.append({'text': f'question: what is the capital of {c}? answer: {cap}.'})
            elif rng.random() < 0.5:
                a, b = rng.randint(1,12), rng.randint(1,12)
                records.append({'text': f'question: what is {a} + {b}? answer: {a+b}.'})
            else:
                records.append({'text': rng.choice([
                    'question: how many days are in a week? answer: 7.',
                    'question: how many months are in a year? answer: 12.',
                    'question: what color is the sky? answer: blue.',
                    'question: what color is grass? answer: green.',
                    'question: what color is blood? answer: red.',
                    'question: what color is snow? answer: white.',
                    'question: what animal says meow? answer: a cat.',
                    'question: what animal says woof? answer: a dog.',
                    'question: what animal says moo? answer: a cow.',
                    'question: what animal says quack? answer: a duck.',
                    'question: what planet do we live on? answer: earth.',
                    'question: what is the closest star to earth? answer: the sun.',
                    'question: how many planets are in the solar system? answer: 8.',
                    'question: what is the largest planet? answer: jupiter.',
                    'question: what is the boiling point of water? answer: 100 degrees celsius.',
                    'question: what is the freezing point of water? answer: 0 degrees celsius.',
                    'question: how many legs does a spider have? answer: 8.',
                    'question: how many legs does an insect have? answer: 6.',
                    'question: how many legs does a dog have? answer: 4.',
                    'question: what is the opposite of hot? answer: cold.',
                    'question: what is the opposite of up? answer: down.',
                    'question: what is the opposite of big? answer: small.',
                    'question: what is the opposite of fast? answer: slow.',
                    'question: what is the opposite of light? answer: dark.',
                    'question: what is the opposite of good? answer: bad.',
                    'question: what is the opposite of day? answer: night.',
                    'question: what is the opposite of wet? answer: dry.',
                    'question: what comes after monday? answer: tuesday.',
                    'question: what comes after friday? answer: saturday.',
                    'question: what comes after december? answer: january.',
                ])})
    return records

# Generate
records = generate_corpus(30000)
Path('/content/einx/data/raw').mkdir(parents=True, exist_ok=True)
with open('/content/einx/data/raw/knowledge.jsonl', 'w') as f:
    for r in records: f.write(json.dumps(r) + '\n')
unique = set(r['text'] for r in records)
print(f'Corpus: {len(records):,} records ({len(unique):,} unique facts)')

# Train tokenizer
tok = BPETokenizer()
tok.train([r['text'] for r in records], vocab_size=1024, verbose=False)
tok.save('/content/einx/data/tokenized/einx-kbpe.json')
print(f'Tokenizer: vocab={tok.vocab_size()}, merges={len(tok.merges)}')

# Build dataset (keep duplicates for memorization)
train_recs, val_recs, _ = train_val_test_split(records, val_ratio=0.05, test_ratio=0.0, seed=42)
out_dir = Path('/content/einx/data/processed')
total_tokens = 0
for split, recs in [('train', train_recs), ('val', val_recs)]:
    d = out_dir / split; d.mkdir(parents=True, exist_ok=True)
    writer = ShardWriter(d, shard_size=10000)
    for rec in recs:
        ids = tok.encode(rec['text'], add_eos=True)[:129]
        writer.write({'input_ids': ids})
        total_tokens += len(ids)
    writer.close()
print(f'Dataset: {len(train_recs):,} train, {len(val_recs):,} val, {total_tokens:,} tokens')

In [ ]:
# ============================================================
# CELL 3: TRAIN THE MODEL ON GPU
# ============================================================
import time, math, json
import torch
from einx.config import EINXModelConfig, TrainingConfig, RuntimeConfig
from einx.data.shards import ShardDataset
from einx.model.transformer import EINXTransformer
from einx.tokenizer.bpe import BPETokenizer
from einx.training.trainer import EINXTrainer
from einx.training.loss_logger import LossLogger

tok = BPETokenizer.load('/content/einx/data/tokenized/einx-kbpe.json')

# Model: ~2.5M params — fits easily on T4 (16GB)
model_cfg = EINXModelConfig(
    name='einx-knowledge-gpu',
    vocab_size=tok.vocab_size(),
    hidden_dim=192,
    n_layers=6,
    n_heads=6,
    head_dim=32,
    max_context_length=128,
    ffn_dim=768,
    dropout=0.1,
    positional_encoding='rope',
    norm_type='rms',
    precision='fp32',       # safe — no AMP issues
    tie_word_embeddings=True,
)

train_cfg = TrainingConfig(
    run_name='knowledge-gpu',
    model_name='einx-knowledge-gpu',
    batch_size=64,
    grad_accum_steps=1,
    learning_rate=0.001,
    weight_decay=0.1,
    max_grad_norm=1.0,
    warmup_steps=100,
    lr_schedule='cosine',
    min_lr_ratio=0.1,
    max_steps=2000,
    save_every_steps=500,
    keep_last_n_checkpoints=3,
    eval_every_steps=200,
    eval_steps=50,
    log_every_steps=100,
    log_level='INFO',
    checkpoint_dir='/content/einx/checkpoints',
    device='cuda',
    precision='fp32',
    seed=42,
)

train_ds = ShardDataset('/content/einx/data/processed/train', context_length=128)
val_ds = ShardDataset('/content/einx/data/processed/val', context_length=128)
model = EINXTransformer(model_cfg)
print(f'Model: {model.n_params:,} params ({model_cfg.n_layers}L {model_cfg.hidden_dim}D {model_cfg.n_heads}H)')
print(f'Data: train={len(train_ds):,}, val={len(val_ds):,}')
print(f'Training: {train_cfg.max_steps} steps on GPU...')
print()

runtime_cfg = RuntimeConfig(device='cuda', precision='fp32')
trainer = EINXTrainer(model, train_cfg, train_ds, val_ds, tokenizer=tok, runtime_config=runtime_cfg)

start = time.time()
result = trainer.train()
elapsed = time.time() - start

entries = LossLogger.load(result.get('loss_log_path',''))
train_e = [e for e in entries if 'val_loss' not in e]
val_e = [e for e in entries if 'val_loss' in e]

print()
print('='*60)
print('TRAINING COMPLETE')
print('='*60)
print(f'Time:       {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'Steps:      {result["final_step"]}')
if train_e:
    print(f'Init loss:  {train_e[0]["loss"]:.4f}')
    print(f'Final loss: {train_e[-1]["loss"]:.4f}')
    print(f'Reduction:  {(1-train_e[-1]["loss"]/train_e[0]["loss"])*100:.1f}%')
if val_e:
    ppl = math.exp(val_e[-1]['val_loss'])
    print(f'Val loss:   {val_e[-1]["val_loss"]:.4f}')
    print(f'Perplexity: {ppl:.2f}')
    print(f'Tokens:     {train_e[-1]["tokens_seen"]:,}')
perf = result.get('performance',{})
if perf:
    print(f'Tok/sec:    {perf.get("avg_tokens_per_second",0):.0f}')
    print(f'Peak mem:   {perf.get("peak_memory_mb",0):.0f} MB')

In [ ]:
# ============================================================
# CELL 4: TEST WITH REAL QUESTIONS
# ============================================================
import torch
from einx.training.checkpoint_manager import CheckpointManager

mgr = CheckpointManager('/content/einx/checkpoints', run_name='knowledge-gpu')
best = mgr.find_best() or mgr.find_latest()
print(f'Checkpoint: {best}')
model = EINXTransformer.load(best, map_location='cuda')
model.eval()

tests = [
    ('question: what is 2 + 2? answer:', '4'),
    ('question: what is 5 + 3? answer:', '8'),
    ('question: what is 7 + 8? answer:', '15'),
    ('question: what is 9 x 9? answer:', '81'),
    ('question: what is 6 x 7? answer:', '42'),
    ('question: what is 10 - 3? answer:', '7'),
    ('question: what is the capital of france? answer:', 'paris'),
    ('question: what is the capital of japan? answer:', 'tokyo'),
    ('question: what is the capital of england? answer:', 'london'),
    ('question: what is the capital of germany? answer:', 'berlin'),
    ('question: what is the capital of italy? answer:', 'rome'),
    ('question: what is the capital of china? answer:', 'beijing'),
    ('question: what is the capital of russia? answer:', 'moscow'),
    ('question: what is the capital of india? answer:', 'delhi'),
    ('question: what planet do we live on? answer:', 'earth'),
    ('question: what is the closest star to earth? answer:', 'sun'),
    ('question: how many days are in a week? answer:', '7'),
    ('question: how many months are in a year? answer:', '12'),
    ('question: what color is the sky? answer:', 'blue'),
    ('question: what color is grass? answer:', 'green'),
    ('question: what color is blood? answer:', 'red'),
    ('question: what animal says meow? answer:', 'cat'),
    ('question: what animal says woof? answer:', 'dog'),
    ('question: what animal says moo? answer:', 'cow'),
    ('question: how many legs does a spider have? answer:', '8'),
    ('question: how many legs does an insect have? answer:', '6'),
    ('question: how many planets are in the solar system? answer:', '8'),
    ('question: what is the largest planet? answer:', 'jupiter'),
    ('question: what is the opposite of hot? answer:', 'cold'),
    ('question: what is the opposite of up? answer:', 'down'),
    ('question: what is the opposite of big? answer:', 'small'),
    ('question: what is the opposite of fast? answer:', 'slow'),
    ('question: what comes after monday? answer:', 'tuesday'),
    ('question: what comes after friday? answer:', 'saturday'),
    ('if today is monday, tomorrow is', 'tuesday'),
    ('if today is friday, tomorrow is', 'saturday'),
    ('if 2x = 10, then x =', '5'),
    ('if x + 5 = 12, then x =', '7'),
]

correct = 0
total = len(tests)
print()
print('='*60)
print('EINX KNOWLEDGE TEST')
print('='*60)
print()

for prompt, expected in tests:
    ids = tok.encode(prompt, add_bos=False)
    input_ids = torch.tensor([ids], dtype=torch.long, device='cuda')
    generated = []
    for _ in range(10):
        with torch.no_grad():
            logits, _ = model(input_ids)
        next_id = torch.argmax(logits[0,-1,:]).item()
        if next_id == tok.special.eos_id: break
        generated.append(next_id)
        input_ids = torch.cat([input_ids, torch.tensor([[next_id]],dtype=torch.long,device='cuda')], dim=1)
    answer = tok.decode(generated).strip().lower()
    is_correct = expected.lower() in answer
    if is_correct: correct += 1
    status = '✓' if is_correct else '✗'
    print(f'{status} Q: {prompt}')
    print(f'  Expected: {expected}  |  Got: {answer!r}')
    print()

print('='*60)
print(f'SCORE: {correct}/{total} ({correct/total*100:.0f}%)')
print('='*60)